# Lesson 8 — full-text search · ค้นด้วยคำ ไม่ใช่ด้วย vector

vector search ตอบว่า "ความหมายใกล้" FTS ตอบว่า "มีคำนี้อยู่จริง"
คนละ index คนละคะแนน `_score` (BM25 ยิ่งสูงยิ่งดี) แทน `_distance` (ยิ่งต่ำยิ่งดี)
โพสต์ 11 โพสต์เดิม บทนี้ไม่แตะ vector เลย

ภาษาไทยไม่มีช่องว่างระหว่างคำ tokenizer เลยสำคัญกว่าปกติ บทนี้ลองสามแบบ

In [1]:
%pip install -q lancedb pandas

Note: you may need to restart the kernel to use updated packages.

In [2]:
import sys, urllib.request, pathlib
if not pathlib.Path("../data/lesson_data.py").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Soul-Brews-Studio/lancedb-oracle/main/lessons/data/lesson_data.py", "lesson_data.py")
sys.path.insert(0, "../data")
from lesson_data import load

import lancedb
from lancedb.index import FTS

db = lancedb.connect("./data")
tbl = db.create_table("posts", data=load("nat_posts.jsonl"), mode="overwrite")

def fts(q):
    df = tbl.search(q, query_type="fts").limit(4).to_pandas()
    return df[["id", "topic", "_score", "text"]].assign(text=lambda d: d.text.str[:42])

[2026-09-10T11:48:22Z WARN  lance::dataset::write::insert] No existing dataset at /opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/08-fts/data/posts.lance, it will be created


**1. tokenizer `simple`** (ค่า default) ตัดที่ช่องว่างและเครื่องหมาย
API ปัจจุบันคือ `create_index(col, config=FTS(...))`
`create_fts_index` ยังเรียกได้แต่ deprecated ตั้งแต่ 0.25

In [3]:
tbl.create_index("text", config=FTS())
fts("Memory")

,id,topic,_score,text
0,p01,memory,2.958381,ยอมกลับมาทำ Memory เพราะซาบซึ้งว่ามันต้องท


คำอังกฤษเจอ แต่คำไทย "จอ" ที่อยู่ใน p09 p10 แน่ ๆ กลับว่างเปล่า
เพราะ "มาออกจอเลยครับ" ทั้งก้อนคือ token เดียว "จอ" ไม่ตรงกับอะไร

In [4]:
fts("จอ")

,id,topic,_score,text


**2. tokenizer `icu`** — ตัดคำด้วย Unicode word segmentation รู้จักภาษาไทย
"มาออกจอเลยครับ" กลายเป็น มา · ออก · จอ · เลย · ครับ
`replace=True` เพราะสร้างทับ index เดิม

In [5]:
tbl.create_index("text", config=FTS(base_tokenizer="icu"), replace=True)
fts("จอ")

,id,topic,_score,text
0,p09,hardware,1.985063,ทำเฟิร์มแวร์เอา Claude Code มาออกจอเลยครับ
1,p10,hardware,1.896931,เอาจอ มาต่อ Claude Code BLE Bridge ลองแล้ว


In [6]:
fts("ความทรงจำ")   # เจอ p03 ตรง ๆ โพสต์เดียว

,id,topic,_score,text
0,p03,memory,8.165496,ความทรงจำระหว่างบรรทัดของ Human และ Claude


**3. tokenizer `ngram`** — หั่นทุกอย่างเป็นชิ้น 2–3 ตัวอักษร ไม่ต้องรู้ภาษา
"ความทรงจำ" เจอ p03 คะแนนสูงมาก แต่ p04 p11 p07 ก็โผล่มาด้วย
เพราะมีชิ้น "ทร" "ควา" ซ้ำกันโดยบังเอิญ ค้นเจอทุกอย่างรวมทั้งขยะ

In [7]:
tbl.create_index("text", config=FTS(base_tokenizer="ngram", ngram_min_length=2, ngram_max_length=3), replace=True)
fts("ความทรงจำ")

,id,topic,_score,text
0,p03,memory,31.619724,ความทรงจำระหว่างบรรทัดของ Human และ Claude
1,p04,memory,9.428171,ทำอินเด็กซ์ Embedding เข้าสู่ Vector Space
2,p11,hardware,5.836645,เตรียมตัวแปลงร่างกันครับ! รอบทความแผ้บบบบ
3,p07,agents,2.226538,เข้าสู่ยุค Multi-Agents แบบเต็มตัว.... Cla


**สรุปในตารางเดียว** — สร้าง index ทั้งสามแบบอีกรอบ ยิงคำถามชุดเดียวกัน
แถว = tokenizer คอลัมน์ = คำค้น ช่อง = id ที่เจอ (— คือไม่เจอ)
`simple` ว่างทุกช่องที่เป็นไทย · `icu` ตรงเป๊ะ · `ngram` เจอเกิน ("ความทรงจำ" ได้ 5 ทั้งที่ควรได้ 1)
สำหรับความจำ agent ที่มีไทยปน `icu` คือค่าที่ควรตั้ง

In [8]:
import pandas as pd

TOKENIZERS = {
    "simple": FTS(),
    "icu": FTS(base_tokenizer="icu"),
    "ngram 2-3": FTS(base_tokenizer="ngram", ngram_min_length=2, ngram_max_length=3),
}
QUERIES = ["Memory", "Claude", "จอ", "ความทรงจำ"]
summary = []
for name, cfg in TOKENIZERS.items():
    tbl.create_index("text", config=cfg, replace=True)
    row = {"tokenizer": name}
    for q in QUERIES:
        ids = [h["id"] for h in tbl.search(q, query_type="fts").limit(5).to_list()]
        row[f'"{q}"'] = " ".join(ids) if ids else "—"
    summary.append(row)
pd.DataFrame(summary)

,tokenizer,"""Memory""","""Claude""","""จอ""","""ความทรงจำ"""
0,simple,p01,p03 p06 p10 p09 p08,—,—
1,icu,p01,p03 p10 p06 p08 p09,p09 p10,p03
2,ngram 2-3,p01 p04,p03 p06 p10 p09 p07,p09 p10,p03 p04 p11 p07 p08


บน disk index อยู่ใน `_indices/` แยกจาก data fragment
สร้าง index ทั้งหมด 6 ครั้ง = 6 directory ของเก่าไม่หาย (Nothing is Deleted) มีแค่อันล่าสุดที่ใช้งาน
`ngram` ใหญ่กว่า `icu` เกือบ 3 เท่า (25.7 KB กับ 9.4 KB) เพราะทุกข้อความแตกเป็นชิ้นเล็กมากกว่า
`add()` แถวใหม่หลังจากนี้ ไม่เข้า index อัตโนมัติ ต้อง `optimize()` (บทที่ 10)

In [9]:
from pathlib import Path
live = {i.name for i in tbl.list_indices()}
dirs = sorted(Path("data/posts.lance/_indices").iterdir(), key=lambda p: p.stat().st_mtime)
labels = ["simple", "icu", "ngram 2-3"] * 2
pd.DataFrame([{
    "order": i + 1,
    "tokenizer": labels[i] if i < len(labels) else "?",
    "index dir": d.name[:8] + "…",
    "bytes": sum(f.stat().st_size for f in d.rglob("*") if f.is_file()),
    "live?": "✓" if i == len(dirs) - 1 else "",
} for i, d in enumerate(dirs)])

,order,tokenizer,index dir,bytes,live?
0,1,simple,38a16cf4…,8923,
1,2,icu,d51ae2a1…,9432,
2,3,ngram 2-3,5b6d4510…,25748,
3,4,simple,c1bca35a…,8923,
4,5,icu,bed5d6bb…,9432,
5,6,ngram 2-3,09ba4783…,25748,✓
